# 06 – Lucene: Indexing, 2 Model Retrieval, Baseline, dan Posting List

| Kode | Konfigurasi | Kelas similarity | Analyzer |
|---|---|---|---|
| **M1** | BM25 + custom preprocessing | `BM25Similarity` | `WhitespaceAnalyzer` pada token hasil `preprocess_text()` |
| **M2** | TF-IDF + custom preprocessing | `ClassicSimilarity` | `WhitespaceAnalyzer` pada token hasil `preprocess_text()` |
| **B1** | **Baseline**: BM25 + `StandardAnalyzer` | `BM25Similarity` | `StandardAnalyzer` pada teks mentah (tanpa preprocessing) |

## 0. Memasang PyLucene

Panduan (Colab dan Linux/WSL) ada di `lucene/README_lucene.md`.

In [ ]:
import importlib, glob, os, subprocess, sys

def pylucene_terpasang():
    importlib.invalidate_caches()
    for egg in glob.glob(os.path.join(sys.prefix, "lib", "python3*", "*-packages", "*.egg")) + \
               glob.glob("/usr/local/lib/python3*/*-packages/*.egg"):
        if os.path.basename(egg).lower().startswith(("lucene", "jcc")) and egg not in sys.path:
            sys.path.insert(0, egg)                      # egg hasil `make install` belum terbaca tanpa restart
    try:
        import lucene
        return True
    except Exception:
        return False

if pylucene_terpasang():
    print("PyLucene sudah terpasang, pemasangan dilewati.")
else:
    VERSI = "10.0.0"
    JAVA_HOME = "/usr/lib/jvm/java-21-openjdk-amd64"
    SUMBER = f"https://archive.apache.org/dist/lucene/pylucene/pylucene-{VERSI}-src.tar.gz"
    env = dict(os.environ, JAVA_HOME=JAVA_HOME, PATH=JAVA_HOME + "/bin:" + os.environ["PATH"], JCC_JDK=JAVA_HOME, NO_SHARED="1")

    def jalankan(perintah, cwd=None):
        print("$", perintah)
        hasil = subprocess.run(perintah, shell=True, cwd=cwd, env=env, capture_output=True, text=True)
        if hasil.returncode != 0:
            print((hasil.stdout + hasil.stderr)[-3000:])
            raise RuntimeError("Langkah gagal: " + perintah)

    # 1) JDK 21 (JDK, bukan JRE), Ant, dan kompiler C++
    jalankan("apt-get -qq update && apt-get -qq install -y openjdk-21-jdk-headless ant build-essential python3-dev curl")
    # 2) JCC mencari libjava/libjvm di jre/lib/amd64; pada JDK modern letaknya di lib/
    jalankan(f"mkdir -p {JAVA_HOME}/jre/lib && ln -sfn ../../lib {JAVA_HOME}/jre/lib/amd64")
    # 3) Unduh kode sumber PyLucene
    jalankan(f"rm -rf /content/pylucene_src && mkdir -p /content/pylucene_src && curl -sSL {SUMBER} | tar -xz --strip-components=1 -C /content/pylucene_src")
    # 4) setuptools modern (pengganti distutils pada Python 3.12 ke atas), lalu bangun dan pasang JCC
    jalankan(f"{sys.executable} -m pip -q install setuptools wheel")
    jalankan(f"{sys.executable} -m pip -q install --no-build-isolation .", cwd="/content/pylucene_src/jcc")
    # 5) Bangun dan pasang PyLucene (proses paling lama)
    jalankan(f"make all install JCC='{sys.executable} -m jcc' ANT=ant PYTHON={sys.executable} NUM_FILES=8", cwd="/content/pylucene_src")

    if pylucene_terpasang():
        print("PyLucene terpasang.")
    else:
        print("PyLucene terpasang, tetapi belum terbaca. Pilih Runtime -> Restart session, lalu jalankan ulang mulai dari sel ini.")

## 1. Memuat korpus dan modul preprocessing kelompok

In [ ]:
import json, os, re, sys, zipfile, math, random, csv, shutil
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:.4f}".format
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

def cari_root():
    ADA = "data/processed/korpus_preprocessed.json"
    for dasar in [Path.cwd(), Path.cwd().parent, Path("/content")]:
        if (dasar / ADA).exists():
            return dasar
        for pola in ("*/" + ADA, "*/*/" + ADA):
            for cocok in sorted(dasar.glob(pola)):
                return cocok.parents[2]
    for ekstrak in [Path("/content/proyek"), Path.cwd() / "proyek"]:
        if ekstrak.exists():
            for cocok in sorted(ekstrak.rglob("korpus_preprocessed.json")):
                if cocok.parent.name == "processed" and cocok.parent.parent.name == "data":
                    return cocok.parents[2]
    return None

ROOT = cari_root()
if ROOT is None:
    try:
        from google.colab import files
        print("Unggah zip proyek kelompok (kelompok4_ForumDiskusiAnimeGame.zip) ...")
        for nama in files.upload():
            if nama.endswith(".zip"):
                zipfile.ZipFile(nama).extractall("/content/proyek" if Path("/content").exists() else "proyek")
        ROOT = cari_root()
    except ImportError:
        pass
assert ROOT is not None, "Folder proyek (data/processed/korpus_preprocessed.json) tidak ditemukan."
print("Folder proyek:", ROOT)

sys.path.insert(0, str(ROOT / "preprocessing"))
ENGINE_DIR = next((p.parent for p in [ROOT / "lucene" / "lucene_engine.py", Path.cwd() / "lucene_engine.py"]
                   if p.exists()), None)
if ENGINE_DIR is None:
    try:
        from google.colab import files
        print("Unggah lucene_engine.py ...")
        files.upload()
        ENGINE_DIR = Path.cwd()
    except ImportError:
        pass
assert ENGINE_DIR is not None, "lucene_engine.py tidak ditemukan."
sys.path.insert(0, str(ENGINE_DIR))

HASIL = ROOT / "lucene" / "hasil"      
HASIL.mkdir(parents=True, exist_ok=True)

from preprocessing import preprocess_text, tokenize, STOPWORDS
with open(ROOT / "data" / "processed" / "korpus_preprocessed.json", encoding="utf-8") as f:
    dokumen = json.load(f)

assert len(dokumen) >= 500, "Korpus kurang dari 500 dokumen."
assert len({d["url"] for d in dokumen}) == len(dokumen), "Ada URL duplikat."
assert all(d["title"].strip() and d["content"].strip() and d["url"] for d in dokumen), "Ada judul/isi/URL kosong."
assert all(d["text_processed"].strip() for d in dokumen), "Ada dokumen yang kosong setelah preprocessing."
print("Jumlah dokumen:", len(dokumen))
print("Per sumber    :", dict(Counter(d["source"] for d in dokumen)))

In [ ]:
acak = random.Random(7).sample(range(len(dokumen)), 300)
beda = []
for i in acak:
    d = dokumen[i]
    ulang = " ".join(p for p in (preprocess_text(d["title"]), preprocess_text(d["content"])) if p)
    if ulang != d["text_processed"]:
        beda.append(i)
print(f"Cocok pada {300 - len(beda)}/300 dokumen sampel.")
assert not beda, f"text_processed berbeda dari preprocess_text() pada dokumen: {beda[:5]}"

## 2. Integrasi preprocessing ke Lucene

Pendekatan yang dipakai : token hasil preprocessing diindeks dengan `WhitespaceAnalyzer`


In [ ]:
import lucene_engine as E

contoh = next(d for d in dokumen if d["source"] == "StackExchange-gaming")
mentah = contoh["title"] + ". " + contoh["content"]
print("JUDUL ASLI     :", contoh["title"])
print("ISI ASLI       :", contoh["content"][:300].replace("\n", " "), "...")
print()
proses = preprocess_text(mentah)
print("SESUDAH PREPROCESSING (preprocess_text):")
print("   ", proses[:300], "...")
print()
sw = E.WhitespaceAnalyzer()
st = E.StandardAnalyzer()
print("Token WhitespaceAnalyzer (M1/M2):", E.token_analyzer(sw, proses)[:20])
print("Token StandardAnalyzer   (B1)   :", E.token_analyzer(st, mentah)[:20])

In [ ]:
kalimat = "I REALLY love these Anime episodes! The players are discussing game bugs & crashes at https://example.com"
print("Asli                      :", kalimat)
print("preprocess_text()         :", preprocess_text(kalimat))
print("Standard (baseline)       :", E.token_analyzer(st, kalimat))
print("Whitespace setelah prepro :", E.token_analyzer(sw, preprocess_text(kalimat)))

## 3. Membangun tiga indeks Lucene

In [ ]:
KONFIG = E.daftar_konfigurasi(preprocess_text)
INDEKS = {kode: E.bangun_indeks(cfg, dokumen, "index_lucene") for kode, cfg in KONFIG.items()}

baris = []
for kode, idx in INDEKS.items():
    s = idx.statistik()
    baris.append({"Kode": kode, "Konfigurasi": idx.konfig.nama, "Similarity": idx.konfig.model,
                  "Analyzer": idx.konfig.analisis, "Dokumen": s["dokumen"], "Term unik": s["term_unik"],
                  "Total token": s["total_token"], "Rata2 token/dok": s["rata2_token_per_dok"]})
tabel_indeks = pd.DataFrame(baris).set_index("Kode")
display(tabel_indeks)
tabel_indeks.to_csv(HASIL / "statistik_indeks.csv", encoding="utf-8-sig")

## 4. Posting list beberapa term

In [ ]:
TERM_CONTOH = ["anim", "game", "rpg", "dub", "crash", "steam"]    
idx1 = INDEKS["M1"]

ringkas = []
for t in TERM_CONTOH:
    p = idx1.posting_list(t, maks=5)
    if p is None:
        ringkas.append({"Term": t, "df": 0, "Total tf": 0, "Posting pertama (doc_id:tf)": "(tidak ada di indeks)"})
        continue
    ringkas.append({"Term": t, "df": p["df"], "Total tf": p["total_tf"],
                    "Posting pertama (doc_id:tf)": ", ".join(f"{d}:{f}" for d, f, _ in p["posting"])})
tabel_posting = pd.DataFrame(ringkas)
display(tabel_posting)
tabel_posting.to_csv(HASIL / "posting_list_ringkas.csv", index=False, encoding="utf-8-sig")

In [ ]:
def tampil_posting(idx, term, maks=8):
    p = idx.posting_list(term, maks=maks)
    print(f"[{idx.konfig.kode}] term '{term}': df={p['df']} dokumen, total kemunculan={p['total_tf']}")
    for doc_id, freq, pos in p["posting"]:
        print(f"   doc {doc_id:>4} | tf={freq:>3} | posisi={pos[:8]}{' ...' if len(pos) > 8 else ''} | {dokumen[doc_id]['title'][:60]}")

tampil_posting(INDEKS["M1"], "rpg")
print()
tampil_posting(INDEKS["B1"], "rpg")

In [ ]:
bandingkan = [("M1", "anim"), ("B1", "anime"), ("B1", "animation"), ("B1", "animated"),
              ("M1", "play"), ("B1", "play"), ("B1", "player"), ("B1", "players"), ("B1", "playing")]
baris = []
for kode, t in bandingkan:
    p = INDEKS[kode].posting_list(t, maks=1)
    baris.append({"Indeks": kode, "Term": t, "df": 0 if p is None else p["df"],
                  "Total tf": 0 if p is None else p["total_tf"]})
display(pd.DataFrame(baris))

## 5. Pencarian dan perbandingan tiga konfigurasi


In [ ]:
def tampil_hasil(query, k=5):
    print(f"QUERY: {query!r}")
    for kode, idx in INDEKS.items():
        print(f"  [{kode}] query ke Lucene: {idx.teks_query(query)!r}")
        for r in idx.cari(query, k):
            print(f"     {r['rank']:>2}. {r['skor']:7.3f}  doc {r['doc_id']:>4}  [{r['sumber']}] {r['judul'][:70]}")
    print()

tampil_hasil("best RPG game")
tampil_hasil("how to fix game crashing on startup")

In [ ]:
QUERY_DEMO = ["best RPG game", "anime sub or dub", "how to fix game crashing on startup",
              "anime with strong female protagonist", "game recommendations for beginners"]
baris = []
for q in QUERY_DEMO:
    top = {kode: [r["doc_id"] for r in idx.cari(q, 10)] for kode, idx in INDEKS.items()}
    baris.append({"Query": q,
                  "Irisan M1∩M2": len(set(top["M1"]) & set(top["M2"])),
                  "Irisan M1∩B1": len(set(top["M1"]) & set(top["B1"])),
                  "Irisan M2∩B1": len(set(top["M2"]) & set(top["B1"]))})
display(pd.DataFrame(baris))

## 6. Rincian skor (`explain`) dan pencocokan dengan rumus

In [ ]:
QUERY_EXPLAIN = "rpg game"
for kode in ["M1", "M2"]:
    idx = INDEKS[kode]
    top = idx.cari(QUERY_EXPLAIN, 1)[0]
    print(f"{kode} | {idx.konfig.model} | dokumen peringkat 1: doc {top['doc_id']} (skor {top['skor']:.4f})")
    print(idx.jelaskan(QUERY_EXPLAIN, top["doc_id"]))
    print()

In [ ]:
def idf_dari_explain(teks, term):
    cocok = re.search(r"weight\(isi:" + re.escape(term) + r" in \d+\).*?\n\s*[\d.E-]+ = score.*?\n\s*([\d.E-]+) = idf", teks, re.S)
    return float(cocok.group(1)) if cocok else float("nan")

baris = []
for kode in ["M1", "M2"]:
    idx = INDEKS[kode]
    for t in ["rpg", "game"]:
        doc = idx.cari(t, 1, query_diproses=t)[0]["doc_id"]
        teks = idx.jelaskan(t, doc, query_diproses=t)
        baris.append({"Model": kode, "Term": t, "n_t (df)": idx.df(t),
                      "idf Lucene": idf_dari_explain(teks, t), "idf rumus": idx.idf_manual(t)})
cek = pd.DataFrame(baris)
display(cek)
assert ((cek["idf Lucene"] - cek["idf rumus"]).abs() < 1e-4).all(), "IDF rumus tidak cocok dengan Lucene."
print("IDF yang dihitung dengan rumus cocok dengan Lucene (selisih < 1e-4).")

## 7. Term vector (Rocchio)
Dua fungsi yang disediakan `lucene_engine.py` untuk Rocchio:

- `Indeks.term_vector(doc_id)` → `{term: tf}`
- `Indeks.cari_berbobot({term: bobot}, k)` → peringkat untuk query berbobot (BooleanQuery + BoostQuery)

In [ ]:
tv = INDEKS["M1"].term_vector(3402)
print("Dokumen 3402:", dokumen[3402]["title"][:80])
print("Jumlah term berbeda pada term vector:", len(tv))
print("10 term dengan tf tertinggi:", sorted(tv.items(), key=lambda x: -x[1])[:10])

bobot = {t: f for t, f in sorted(tv.items(), key=lambda x: -x[1])[:10]}
for r in INDEKS["M1"].cari_berbobot(bobot, 3):
    print(f"   {r['rank']}. {r['skor']:.3f} doc {r['doc_id']} {r['judul'][:70]}")

## 8. Penilaian relevansi manual untuk Q6–Q7

In [ ]:
QUERIES = {
    "Q1": "what order should I watch an anime series and its movies",
    "Q2": "where to watch english dubbed anime",
    "Q3": "difference between anime and manga story",
    "Q4": "how to fix game crashing on startup",
    "Q5": "game freezing or lagging with low fps",
    "Q6": "isekai anime",
    "Q7": "where are save files and cloud saves for games",
    "Q8": "how to install mods for a game",
    "Q9": "trade pokemon between games",
    "Q10": "voice actor of a character",
}
MILIKKU = ["Q6", "Q7"]
BERKAS_NILAI = E.buat_pool_penilaian(INDEKS, dokumen, QUERIES, MILIKKU, HASIL)

In [ ]:
try:
    from google.colab import files
    for nama in files.upload():
        if nama.startswith("penilaian_Q6_Q7"):
            shutil.copy(nama, HASIL / "penilaian_Q6_Q7.csv")
            print("Disalin ke", HASIL / "penilaian_Q6_Q7.csv")
except ImportError:
    print("Bukan Colab: letakkan CSV yang sudah diisi di", HASIL)

In [ ]:
relevansi = E.muat_penilaian(HASIL, MILIKKU)

## 10. Ekspor peringkat untuk evaluasi

Agar evaluasi tidak perlu memasang PyLucene, peringkat 100 teratas tiap query untuk ketiga konfigurasi disimpan ke `ranking_semua_query.json` (bentuk: `{kode: {qid: [[doc_id, skor], ...]}}`). Precision@10, Recall, F1, dan MAP dihitung dari file ini bersama penilaian relevansi semua anggota. Pool Q6–Q7 sebelumnya memakai kedalaman 10; bila evaluasi memakai kedalaman lebih dalam untuk Recall/MAP, dokumen di peringkat 11–100 yang tidak dinilai dianggap tidak relevan.

In [ ]:
ranking = E.ekspor_ranking(INDEKS, QUERIES, HASIL)
print({kode: len(per) for kode, per in ranking.items()}, "query per konfigurasi")

In [ ]:
arsip = E.kemas_hasil(HASIL)